# 임베딩 모델 평가 테스트

RAG 기반 서비스에서 사용할 임베딩 모델 후보 4종을 비교 평가합니다.

## 후보 모델
1. **Gemini Embedding** (`gemini-embedding-001`) - 기본 3072차원 (768로 축소 가능), API 기반
2. **ko-sbert-multitask** (`jhgan/ko-sbert-multitask`) - 768차원, 로컬
3. **OpenAI Embedding** (`text-embedding-3-small`) - 1536차원, API 기반
4. **BGE-M3** (`BAAI/bge-m3`) - 1024차원, 로컬

## 평가 기준
- 한국어 의미 유사도 (Cosine Similarity)
- RAG 비대칭 검색 성능 (짧은 쿼리 ↔ 긴 문서)
- 추론 속도
- task_type 지원 여부

> **참고**: `text-embedding-004`는 2026-01-14부로 deprecated되었으며, `gemini-embedding-001`로 교체됨

---
## 0. 환경 설정 및 패키지 설치

In [ ]:
!pip install -q google-genai openai sentence-transformers transformers torch numpy scikit-learn pandas

### API 키 입력

아래 셀에서 직접 API 키를 입력하세요.

In [ ]:
from getpass import getpass

# ========================================
# API 키 입력 (실행 시 입력 프롬프트 표시)
# ========================================
GOOGLE_API_KEY = getpass("Google AI API Key 입력: ")
OPENAI_API_KEY = getpass("OpenAI API Key 입력: ")
HF_TOKEN = getpass("Hugging Face Token 입력 (없으면 Enter): ")

print("✅ API 키 설정 완료")

### 공통 유틸리티

In [ ]:
import numpy as np
import time
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

def calc_cosine_sim(vec_a, vec_b):
    """두 벡터 간 코사인 유사도 계산"""
    a = np.array(vec_a).reshape(1, -1)
    b = np.array(vec_b).reshape(1, -1)
    return cosine_similarity(a, b)[0][0]

def measure_time(func, *args, **kwargs):
    """함수 실행 시간 측정 (ms)"""
    start = time.time()
    result = func(*args, **kwargs)
    elapsed = (time.time() - start) * 1000
    return result, elapsed

### 테스트 데이터셋

RAG 비대칭 검색 시나리오 (짧은 쿼리 ↔ 긴 문서)와 대칭 유사도 비교 시나리오를 포함합니다.

In [ ]:
# =============================================
# 1) RAG 비대칭 검색 테스트 데이터 (쿼리 ↔ 문서)
# =============================================
rag_test_data = [
    {
        "query": "백엔드 개발자 이력서 분석해줘",
        "relevant_doc": "3년 경력의 백엔드 개발자입니다. Java, Spring Boot, MySQL을 주로 사용하며 대규모 트래픽 처리 경험이 있습니다. AWS 환경에서 마이크로서비스 아키텍처를 설계하고 운영한 경험이 있으며, JPA와 QueryDSL을 활용한 데이터베이스 최적화 작업을 수행했습니다.",
        "irrelevant_doc": "오늘 서울의 날씨는 맑고 기온은 영하 5도입니다. 미세먼지 농도는 보통 수준이며, 오후부터 구름이 끼기 시작할 것으로 예상됩니다."
    },
    {
        "query": "프론트엔드 React 경험자 찾아줘",
        "relevant_doc": "프론트엔드 개발자로 5년간 근무했습니다. React, TypeScript, Next.js를 활용한 SPA 개발이 주 업무이며, Redux와 React Query를 사용한 상태관리 경험이 풍부합니다. Figma 디자인을 기반으로 반응형 UI를 구현하고, Jest와 Cypress로 테스트 자동화를 적용했습니다.",
        "irrelevant_doc": "강남역 맛집 추천입니다. 스시 오마카세, 이탈리안 레스토랑, 한정식 등 다양한 음식점이 있으며, 예약은 필수입니다."
    },
    {
        "query": "데이터 엔지니어링 포트폴리오",
        "relevant_doc": "데이터 파이프라인 구축 프로젝트를 수행했습니다. Apache Airflow로 ETL 워크플로우를 자동화하고, Spark와 Kafka를 활용하여 실시간 데이터 처리 시스템을 구현했습니다. BigQuery와 Redshift를 활용한 데이터 웨어하우스 설계 경험도 있습니다.",
        "irrelevant_doc": "올해 인기 있는 넷플릭스 드라마 목록입니다. 로맨스, 스릴러, SF 장르 등 다양한 장르의 콘텐츠가 있습니다."
    }
]

# =============================================
# 2) 대칭 유사도 비교 테스트 데이터 (문장 ↔ 문장)
# =============================================
symmetric_test_data = [
    {
        "sent_a": "오늘 날씨가 정말 좋다",
        "sent_b": "날씨가 화창하고 맑네요",
        "expected": "high"  # 높은 유사도 기대
    },
    {
        "sent_a": "Python으로 웹 개발을 합니다",
        "sent_b": "파이썬을 사용하여 웹 서비스를 만듭니다",
        "expected": "high"
    },
    {
        "sent_a": "오늘 날씨가 정말 좋다",
        "sent_b": "Python으로 웹 개발을 합니다",
        "expected": "low"  # 낮은 유사도 기대
    },
    {
        "sent_a": "이력서를 작성하고 있습니다",
        "sent_b": "자기소개서를 쓰는 중이에요",
        "expected": "high"
    }
]

print(f"RAG 비대칭 테스트: {len(rag_test_data)}건")
print(f"대칭 유사도 테스트: {len(symmetric_test_data)}건")

---
## 1. Gemini Embedding (`gemini-embedding-001`)

- 기본 3072차원 (output_dimensionality로 768 축소 가능)
- 새로운 `google-genai` SDK 사용
- `task_type` 지원: RETRIEVAL_QUERY / RETRIEVAL_DOCUMENT / SEMANTIC_SIMILARITY 등
- RAG 비대칭 검색에 최적화
- Matryoshka Representation Learning (MRL) 지원

In [ ]:
from google import genai
from google.genai import types

# Gemini 클라이언트 초기화
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

# 프로젝트에서 사용하는 768차원으로 맞춤 (기본값은 3072)
GEMINI_OUTPUT_DIM = 768

def gemini_embed(text, task_type="RETRIEVAL_DOCUMENT"):
    """Gemini Embedding API를 사용한 임베딩 생성 (새로운 google-genai SDK)"""
    result = gemini_client.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
        config=types.EmbedContentConfig(
            task_type=task_type,
            output_dimensionality=GEMINI_OUTPUT_DIM
        )
    )
    return result.embeddings[0].values

# 기본 테스트
test_emb = gemini_embed("테스트 문장입니다")
print(f"Gemini Embedding 차원: {len(test_emb)}")
print(f"벡터 샘플 (처음 5개): {test_emb[:5]}")

### 1-1. Gemini - RAG 비대칭 검색 테스트

`task_type`을 `retrieval_query`(쿼리)와 `retrieval_document`(문서)로 구분하여 검색 성능을 테스트합니다.

In [ ]:
print("=" * 60)
print("Gemini Embedding - RAG 비대칭 검색 테스트")
print("=" * 60)

gemini_rag_results = []

for i, test in enumerate(rag_test_data):
    # 쿼리는 RETRIEVAL_QUERY, 문서는 RETRIEVAL_DOCUMENT로 임베딩
    query_emb, q_time = measure_time(
        gemini_embed, test["query"], task_type="RETRIEVAL_QUERY"
    )
    rel_emb, d_time = measure_time(
        gemini_embed, test["relevant_doc"], task_type="RETRIEVAL_DOCUMENT"
    )
    irr_emb, _ = measure_time(
        gemini_embed, test["irrelevant_doc"], task_type="RETRIEVAL_DOCUMENT"
    )

    rel_sim = calc_cosine_sim(query_emb, rel_emb)
    irr_sim = calc_cosine_sim(query_emb, irr_emb)
    gap = rel_sim - irr_sim

    gemini_rag_results.append({
        "query": test["query"],
        "relevant_sim": rel_sim,
        "irrelevant_sim": irr_sim,
        "gap": gap,
        "query_time_ms": q_time,
        "doc_time_ms": d_time
    })

    print(f"\n[테스트 {i+1}] 쿼리: {test['query']}")
    print(f"  관련 문서 유사도: {rel_sim:.4f}")
    print(f"  비관련 문서 유사도: {irr_sim:.4f}")
    print(f"  Gap (클수록 좋음): {gap:.4f}")
    print(f"  쿼리 임베딩 시간: {q_time:.1f}ms")
    print(f"  문서 임베딩 시간: {d_time:.1f}ms")

# 평균
avg_rel = np.mean([r["relevant_sim"] for r in gemini_rag_results])
avg_irr = np.mean([r["irrelevant_sim"] for r in gemini_rag_results])
avg_gap = np.mean([r["gap"] for r in gemini_rag_results])
avg_time = np.mean([r["query_time_ms"] for r in gemini_rag_results])

print(f"\n{'=' * 60}")
print(f"[Gemini 평균] 관련: {avg_rel:.4f} | 비관련: {avg_irr:.4f} | Gap: {avg_gap:.4f} | 속도: {avg_time:.1f}ms")

### 1-2. Gemini - 대칭 유사도 테스트

In [ ]:
print("=" * 60)
print("Gemini Embedding - 대칭 유사도 테스트")
print("=" * 60)

gemini_sym_results = []

for i, test in enumerate(symmetric_test_data):
    emb_a, t_a = measure_time(
        gemini_embed, test["sent_a"], task_type="SEMANTIC_SIMILARITY"
    )
    emb_b, t_b = measure_time(
        gemini_embed, test["sent_b"], task_type="SEMANTIC_SIMILARITY"
    )

    sim = calc_cosine_sim(emb_a, emb_b)
    gemini_sym_results.append({
        "sent_a": test["sent_a"],
        "sent_b": test["sent_b"],
        "similarity": sim,
        "expected": test["expected"]
    })

    status = "O" if (sim > 0.7 and test["expected"] == "high") or (sim < 0.5 and test["expected"] == "low") else "X"
    print(f"\n[테스트 {i+1}] [{status}] 유사도: {sim:.4f} (기대: {test['expected']})")
    print(f"  A: {test['sent_a']}")
    print(f"  B: {test['sent_b']}")

---
## 2. ko-sbert-multitask (`jhgan/ko-sbert-multitask`)

- 768차원, 로컬 실행
- 한국어 특화 SBERT
- `task_type` 미지원 (대칭 유사도 전용)
- HuggingFace에서 다운로드

In [ ]:
from sentence_transformers import SentenceTransformer
import os

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

print("ko-sbert-multitask 모델 로딩 중...")
ko_sbert_model = SentenceTransformer('jhgan/ko-sbert-multitask')
print(f"모델 로딩 완료!")

def kosbert_embed(text):
    """ko-sbert를 사용한 임베딩 생성"""
    return ko_sbert_model.encode(text).tolist()

# 기본 테스트
test_emb = kosbert_embed("테스트 문장입니다")
print(f"ko-sbert 임베딩 차원: {len(test_emb)}")
print(f"벡터 샘플 (처음 5개): {test_emb[:5]}")

### 2-1. ko-sbert - RAG 비대칭 검색 테스트

ko-sbert는 `task_type`을 지원하지 않으므로 쿼리와 문서를 동일하게 처리합니다.

In [ ]:
print("=" * 60)
print("ko-sbert-multitask - RAG 비대칭 검색 테스트")
print("(task_type 미지원 - 쿼리/문서 동일 처리)")
print("=" * 60)

kosbert_rag_results = []

for i, test in enumerate(rag_test_data):
    query_emb, q_time = measure_time(kosbert_embed, test["query"])
    rel_emb, d_time = measure_time(kosbert_embed, test["relevant_doc"])
    irr_emb, _ = measure_time(kosbert_embed, test["irrelevant_doc"])

    rel_sim = calc_cosine_sim(query_emb, rel_emb)
    irr_sim = calc_cosine_sim(query_emb, irr_emb)
    gap = rel_sim - irr_sim

    kosbert_rag_results.append({
        "query": test["query"],
        "relevant_sim": rel_sim,
        "irrelevant_sim": irr_sim,
        "gap": gap,
        "query_time_ms": q_time,
        "doc_time_ms": d_time
    })

    print(f"\n[테스트 {i+1}] 쿼리: {test['query']}")
    print(f"  관련 문서 유사도: {rel_sim:.4f}")
    print(f"  비관련 문서 유사도: {irr_sim:.4f}")
    print(f"  Gap (클수록 좋음): {gap:.4f}")
    print(f"  쿼리 임베딩 시간: {q_time:.1f}ms")
    print(f"  문서 임베딩 시간: {d_time:.1f}ms")

avg_rel = np.mean([r["relevant_sim"] for r in kosbert_rag_results])
avg_irr = np.mean([r["irrelevant_sim"] for r in kosbert_rag_results])
avg_gap = np.mean([r["gap"] for r in kosbert_rag_results])
avg_time = np.mean([r["query_time_ms"] for r in kosbert_rag_results])

print(f"\n{'=' * 60}")
print(f"[ko-sbert 평균] 관련: {avg_rel:.4f} | 비관련: {avg_irr:.4f} | Gap: {avg_gap:.4f} | 속도: {avg_time:.1f}ms")

### 2-2. ko-sbert - 대칭 유사도 테스트

In [ ]:
print("=" * 60)
print("ko-sbert-multitask - 대칭 유사도 테스트")
print("=" * 60)

kosbert_sym_results = []

for i, test in enumerate(symmetric_test_data):
    emb_a, t_a = measure_time(kosbert_embed, test["sent_a"])
    emb_b, t_b = measure_time(kosbert_embed, test["sent_b"])

    sim = calc_cosine_sim(emb_a, emb_b)
    kosbert_sym_results.append({
        "sent_a": test["sent_a"],
        "sent_b": test["sent_b"],
        "similarity": sim,
        "expected": test["expected"]
    })

    status = "O" if (sim > 0.7 and test["expected"] == "high") or (sim < 0.5 and test["expected"] == "low") else "X"
    print(f"\n[테스트 {i+1}] [{status}] 유사도: {sim:.4f} (기대: {test['expected']})")
    print(f"  A: {test['sent_a']}")
    print(f"  B: {test['sent_b']}")

---
## 3. OpenAI Embedding (`text-embedding-3-small`)

- 1536차원, API 기반
- 한국어 성능 우수
- Gemini보다 2배 비용

In [ ]:
from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)

def openai_embed(text):
    """OpenAI Embedding API를 사용한 임베딩 생성"""
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# 기본 테스트
test_emb = openai_embed("테스트 문장입니다")
print(f"OpenAI Embedding 차원: {len(test_emb)}")
print(f"벡터 샘플 (처음 5개): {test_emb[:5]}")

### 3-1. OpenAI - RAG 비대칭 검색 테스트

OpenAI Embedding은 별도 task_type 없이 단일 모델로 처리합니다.

In [ ]:
print("=" * 60)
print("OpenAI text-embedding-3-small - RAG 비대칭 검색 테스트")
print("=" * 60)

openai_rag_results = []

for i, test in enumerate(rag_test_data):
    query_emb, q_time = measure_time(openai_embed, test["query"])
    rel_emb, d_time = measure_time(openai_embed, test["relevant_doc"])
    irr_emb, _ = measure_time(openai_embed, test["irrelevant_doc"])

    rel_sim = calc_cosine_sim(query_emb, rel_emb)
    irr_sim = calc_cosine_sim(query_emb, irr_emb)
    gap = rel_sim - irr_sim

    openai_rag_results.append({
        "query": test["query"],
        "relevant_sim": rel_sim,
        "irrelevant_sim": irr_sim,
        "gap": gap,
        "query_time_ms": q_time,
        "doc_time_ms": d_time
    })

    print(f"\n[테스트 {i+1}] 쿼리: {test['query']}")
    print(f"  관련 문서 유사도: {rel_sim:.4f}")
    print(f"  비관련 문서 유사도: {irr_sim:.4f}")
    print(f"  Gap (클수록 좋음): {gap:.4f}")
    print(f"  쿼리 임베딩 시간: {q_time:.1f}ms")
    print(f"  문서 임베딩 시간: {d_time:.1f}ms")

avg_rel = np.mean([r["relevant_sim"] for r in openai_rag_results])
avg_irr = np.mean([r["irrelevant_sim"] for r in openai_rag_results])
avg_gap = np.mean([r["gap"] for r in openai_rag_results])
avg_time = np.mean([r["query_time_ms"] for r in openai_rag_results])

print(f"\n{'=' * 60}")
print(f"[OpenAI 평균] 관련: {avg_rel:.4f} | 비관련: {avg_irr:.4f} | Gap: {avg_gap:.4f} | 속도: {avg_time:.1f}ms")

### 3-2. OpenAI - 대칭 유사도 테스트

In [ ]:
print("=" * 60)
print("OpenAI text-embedding-3-small - 대칭 유사도 테스트")
print("=" * 60)

openai_sym_results = []

for i, test in enumerate(symmetric_test_data):
    emb_a, t_a = measure_time(openai_embed, test["sent_a"])
    emb_b, t_b = measure_time(openai_embed, test["sent_b"])

    sim = calc_cosine_sim(emb_a, emb_b)
    openai_sym_results.append({
        "sent_a": test["sent_a"],
        "sent_b": test["sent_b"],
        "similarity": sim,
        "expected": test["expected"]
    })

    status = "O" if (sim > 0.7 and test["expected"] == "high") or (sim < 0.5 and test["expected"] == "low") else "X"
    print(f"\n[테스트 {i+1}] [{status}] 유사도: {sim:.4f} (기대: {test['expected']})")
    print(f"  A: {test['sent_a']}")
    print(f"  B: {test['sent_b']}")

---
## 4. BGE-M3 (`BAAI/bge-m3`)

- 1024차원, 오픈소스 로컬 실행
- 다국어 지원
- GPU 권장 (Colab GPU 런타임 사용 권장)
- HuggingFace에서 다운로드

In [ ]:
import os

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

print("BGE-M3 모델 로딩 중... (모델 크기가 크므로 시간이 걸릴 수 있습니다)")
bge_model = SentenceTransformer('BAAI/bge-m3')
print("모델 로딩 완료!")

def bge_embed(text):
    """BGE-M3를 사용한 임베딩 생성"""
    return bge_model.encode(text).tolist()

# 기본 테스트
test_emb = bge_embed("테스트 문장입니다")
print(f"BGE-M3 임베딩 차원: {len(test_emb)}")
print(f"벡터 샘플 (처음 5개): {test_emb[:5]}")

### 4-1. BGE-M3 - RAG 비대칭 검색 테스트

In [ ]:
print("=" * 60)
print("BGE-M3 - RAG 비대칭 검색 테스트")
print("=" * 60)

bge_rag_results = []

for i, test in enumerate(rag_test_data):
    query_emb, q_time = measure_time(bge_embed, test["query"])
    rel_emb, d_time = measure_time(bge_embed, test["relevant_doc"])
    irr_emb, _ = measure_time(bge_embed, test["irrelevant_doc"])

    rel_sim = calc_cosine_sim(query_emb, rel_emb)
    irr_sim = calc_cosine_sim(query_emb, irr_emb)
    gap = rel_sim - irr_sim

    bge_rag_results.append({
        "query": test["query"],
        "relevant_sim": rel_sim,
        "irrelevant_sim": irr_sim,
        "gap": gap,
        "query_time_ms": q_time,
        "doc_time_ms": d_time
    })

    print(f"\n[테스트 {i+1}] 쿼리: {test['query']}")
    print(f"  관련 문서 유사도: {rel_sim:.4f}")
    print(f"  비관련 문서 유사도: {irr_sim:.4f}")
    print(f"  Gap (클수록 좋음): {gap:.4f}")
    print(f"  쿼리 임베딩 시간: {q_time:.1f}ms")
    print(f"  문서 임베딩 시간: {d_time:.1f}ms")

avg_rel = np.mean([r["relevant_sim"] for r in bge_rag_results])
avg_irr = np.mean([r["irrelevant_sim"] for r in bge_rag_results])
avg_gap = np.mean([r["gap"] for r in bge_rag_results])
avg_time = np.mean([r["query_time_ms"] for r in bge_rag_results])

print(f"\n{'=' * 60}")
print(f"[BGE-M3 평균] 관련: {avg_rel:.4f} | 비관련: {avg_irr:.4f} | Gap: {avg_gap:.4f} | 속도: {avg_time:.1f}ms")

### 4-2. BGE-M3 - 대칭 유사도 테스트

In [ ]:
print("=" * 60)
print("BGE-M3 - 대칭 유사도 테스트")
print("=" * 60)

bge_sym_results = []

for i, test in enumerate(symmetric_test_data):
    emb_a, t_a = measure_time(bge_embed, test["sent_a"])
    emb_b, t_b = measure_time(bge_embed, test["sent_b"])

    sim = calc_cosine_sim(emb_a, emb_b)
    bge_sym_results.append({
        "sent_a": test["sent_a"],
        "sent_b": test["sent_b"],
        "similarity": sim,
        "expected": test["expected"]
    })

    status = "O" if (sim > 0.7 and test["expected"] == "high") or (sim < 0.5 and test["expected"] == "low") else "X"
    print(f"\n[테스트 {i+1}] [{status}] 유사도: {sim:.4f} (기대: {test['expected']})")
    print(f"  A: {test['sent_a']}")
    print(f"  B: {test['sent_b']}")

---
## 5. 전체 모델 비교 결과

In [ ]:
# =============================================
# RAG 비대칭 검색 비교
# =============================================
print("=" * 80)
print("RAG 비대칭 검색 성능 비교 (쿼리 ↔ 문서)")
print("=" * 80)

all_rag_results = {
    "Gemini Embedding": gemini_rag_results,
    "ko-sbert-multitask": kosbert_rag_results,
    "OpenAI Embedding": openai_rag_results,
    "BGE-M3": bge_rag_results
}

rag_summary = []
for model_name, results in all_rag_results.items():
    rag_summary.append({
        "모델": model_name,
        "평균 관련 유사도": np.mean([r["relevant_sim"] for r in results]),
        "평균 비관련 유사도": np.mean([r["irrelevant_sim"] for r in results]),
        "평균 Gap": np.mean([r["gap"] for r in results]),
        "평균 속도(ms)": np.mean([r["query_time_ms"] for r in results]),
        "task_type 지원": "O" if model_name == "Gemini Embedding" else "X"
    })

df_rag = pd.DataFrame(rag_summary)
df_rag = df_rag.sort_values("평균 Gap", ascending=False)
print(df_rag.to_string(index=False, float_format="{:.4f}".format))

print(f"\n>> Gap이 클수록 관련 문서와 비관련 문서를 잘 구분합니다.")
print(f">> RAG에서는 Gap이 핵심 지표입니다.")

In [ ]:
# =============================================
# 대칭 유사도 비교
# =============================================
print("=" * 80)
print("대칭 유사도 비교 (문장 ↔ 문장)")
print("=" * 80)

all_sym_results = {
    "Gemini Embedding": gemini_sym_results,
    "ko-sbert-multitask": kosbert_sym_results,
    "OpenAI Embedding": openai_sym_results,
    "BGE-M3": bge_sym_results
}

sym_summary = []
for model_name, results in all_sym_results.items():
    high_sims = [r["similarity"] for r in results if r["expected"] == "high"]
    low_sims = [r["similarity"] for r in results if r["expected"] == "low"]
    sym_summary.append({
        "모델": model_name,
        "유사 문장 평균": np.mean(high_sims),
        "비유사 문장 평균": np.mean(low_sims),
        "구분 Gap": np.mean(high_sims) - np.mean(low_sims)
    })

df_sym = pd.DataFrame(sym_summary)
df_sym = df_sym.sort_values("구분 Gap", ascending=False)
print(df_sym.to_string(index=False, float_format="{:.4f}".format))

In [ ]:
# =============================================
# 종합 비교표
# =============================================
print("=" * 80)
print("종합 비교표")
print("=" * 80)

final_summary = []
model_info = {
    "Gemini Embedding": {"차원": GEMINI_OUTPUT_DIM, "비용": "$0.00001/1K tok", "타입": "API", "task_type": "O"},
    "ko-sbert-multitask": {"차원": 768, "비용": "무료 (로컬)", "타입": "로컬", "task_type": "X"},
    "OpenAI Embedding": {"차원": 1536, "비용": "$0.00002/1K tok", "타입": "API", "task_type": "X"},
    "BGE-M3": {"차원": 1024, "비용": "무료 (로컬)", "타입": "로컬", "task_type": "X"}
}

for model_name in all_rag_results.keys():
    rag_r = all_rag_results[model_name]
    sym_r = all_sym_results[model_name]
    info = model_info[model_name]

    high_sims = [r["similarity"] for r in sym_r if r["expected"] == "high"]

    final_summary.append({
        "모델": model_name,
        "차원": info["차원"],
        "타입": info["타입"],
        "비용": info["비용"],
        "task_type": info["task_type"],
        "RAG Gap": f"{np.mean([r['gap'] for r in rag_r]):.4f}",
        "유사도 정확도": f"{np.mean(high_sims):.4f}",
        "평균 속도(ms)": f"{np.mean([r['query_time_ms'] for r in rag_r]):.1f}"
    })

df_final = pd.DataFrame(final_summary)
print(df_final.to_string(index=False))

print("\n" + "=" * 80)
print("결론")
print("=" * 80)
print("""
RAG 기반 서비스 관점:
  - RAG Gap (관련 vs 비관련 문서 구분력)이 핵심 지표
  - task_type 지원 여부가 비대칭 검색 성능에 영향
  - Gemini gemini-embedding-001은 MRL 지원으로 768/1536/3072 차원 선택 가능

비용 관점:
  - Gemini: 거의 무료 ($0.00001/1K tokens)
  - OpenAI: Gemini 대비 2배
  - ko-sbert/BGE-M3: 무료 (서버 리소스 필요)

모델 변경 참고:
  - text-embedding-004 → gemini-embedding-001 (2026-01-14 deprecated)
  - google.generativeai → google-genai SDK로 마이그레이션 필요
""")